# Task 2 — Gestalt Principles of Grouping

**Data:** `stimuli_gestalt.csv` — 284 rows across six panels, one per Gestalt principle.

The data is deliberately near-identical across panels. What changes is the *arrangement*, so any grouping a reader perceives comes from the layout, never from the numbers.

**Participant:** a classmate (Farhan), shown panels on a laptop screen, asked panel by panel: 'how many groups do you see?'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

gdf = pd.read_csv('data/stimuli_gestalt.csv')
tips = pd.read_csv('data/tips.csv')
print('Gestalt panels:', sorted(gdf['panel'].unique()))
print('Rows per panel:')
print(gdf['panel'].value_counts())
gdf.head()

In [ ]:
# ── Step 1: Draw all six panels in a 2×3 figure ───────────────────────────
panels = ['proximity', 'similarity', 'enclosure', 'continuity', 'closure', 'connection']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes_flat = axes.flatten()

for ax, panel in zip(axes_flat, panels):
    sub = gdf[gdf['panel'] == panel]

    if panel == 'connection':
        # Draw connecting lines first (behind dots)
        for connector_id, pair in sub.groupby('connector'):
            if len(pair) == 2:
                pts = pair[['x', 'y']].values
                ax.plot(pts[:, 0], pts[:, 1], color='#aaaaaa', linewidth=1.2,
                        zorder=1)

    # Plot items
    for colour, grp in sub.groupby('colour'):
        ax.scatter(grp['x'], grp['y'],
                   color=colour, s=grp['size'].iloc[0],
                   edgecolors='white', linewidths=0.5, zorder=2)

    if panel == 'enclosure':
        # Add four rectangular boundaries (grey edge, no fill)
        box_origins = [(0, 0), (5, 0), (0, 5), (5, 5)]
        for (ox, oy) in box_origins:
            rect = mpatches.Rectangle(
                (ox, oy), 3, 3,
                linewidth=1.5, edgecolor='#888888', facecolor='none'
            )
            ax.add_patch(rect)

    ax.set_title(panel.capitalize(), fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle('Six Gestalt principles — each panel uses the same items; only the arrangement differs',
             fontsize=13, x=0.01, ha='left')
plt.tight_layout()
plt.savefig('gestalt_panels.png', dpi=300, bbox_inches='tight')
plt.show()

**What the eye does in each panel:**

- **Proximity:** identical grey circles are clumped into four spatially separated groups. The brain infers that *things placed close together belong together* — proximity. No colour or shape difference exists; space alone signals group membership.
- **Similarity:** items spread uniformly across the canvas are grouped by colour alone. The brain infers that *things that look alike belong together* — similarity. Spatial position gives no information.
- **Enclosure:** four rectangles tell the brain that the items inside each boundary form a unit — *things inside a shared boundary belong together*. The enclosure overrides the fact that items across adjacent groups may be closer than items within the same group.
- **Continuity:** two paths cross in the middle. The brain follows each smooth curve through the intersection rather than switching paths — *items along a smooth path belong to that path* (good continuation).
- **Closure:** two arcs with a deliberate gap. The brain completes the shapes into full circles — *an incomplete familiar shape is completed*. The gap is the stimulus; 'fixing' it would destroy the demonstration.
- **Connection:** pairs of circles at different colours joined by a grey line. Connection beats both proximity (partners are not necessarily nearest) and similarity (partners are different colours) — *things joined by a line belong together*.

In [ ]:
# ── Step 2: Human perception table ──────────────────────────────────────────
true_groups = {'proximity': 4, 'similarity': 4, 'enclosure': 4,
               'continuity': 2, 'closure': 2, 'connection': 24}

# Farhan's answers (recorded verbatim during the session)
farhan_answers = {'proximity': 4, 'similarity': 4, 'enclosure': 4,
                  'continuity': 2, 'closure': 2, 'connection': 21}

table = pd.DataFrame({
    'panel':           list(true_groups.keys()),
    'principle':       ['Proximity', 'Similarity', 'Enclosure',
                        'Continuity', 'Closure', 'Connection'],
    'true_groups':     list(true_groups.values()),
    'perceived_groups': list(farhan_answers.values()),
})
table['match'] = table['true_groups'] == table['perceived_groups']
table['match'] = table['match'].map({True: 'yes', False: 'no'})

print(table.to_string(index=False))
print('\nNote: Connection panel — Farhan saw 21 instead of 24 pairs.')
print('He said several pairs looked like three items in a chain rather than separate pairs.')
print('This is consistent with real perception: proximity of nearby connectors creates ambiguity.')

In [ ]:
# ── Step 3: Make the principles fight — connection with vs without lines ────
conn = gdf[gdf['panel'] == 'connection'].copy()

fig, (ax_with, ax_without) = plt.subplots(1, 2, figsize=(14, 6))

for ax, show_lines in [(ax_with, True), (ax_without, False)]:
    if show_lines:
        for connector_id, pair in conn.groupby('connector'):
            if len(pair) == 2:
                pts = pair[['x', 'y']].values
                ax.plot(pts[:, 0], pts[:, 1],
                        color='#aaaaaa', linewidth=1.2, zorder=1)

    for colour, grp in conn.groupby('colour'):
        ax.scatter(grp['x'], grp['y'],
                   color=colour, s=60,
                   edgecolors='white', linewidths=0.5, zorder=2)

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

ax_with.set_title('WITH connecting lines\nConnection beats similarity and proximity', loc='left')
ax_without.set_title('WITHOUT connecting lines\nSimilarity (colour) takes over', loc='left')

fig.suptitle('Connection vs similarity and proximity — which principle wins?', x=0.02, ha='left')
plt.tight_layout()
plt.savefig('connection_fight.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Farhan's responses:
  WITH lines:    'I see pairs — they're joined by lines, even though they're different colours.'
                 → Grouped by connection (24 pairs perceived as ~21 pairs)
  WITHOUT lines: 'I see two groups — the blue ones and the red ones.'
                 → Grouped by similarity (colour) — connection is gone, and similarity wins over proximity

Result: connection beat both similarity and proximity, as predicted.
Without the lines, similarity (colour) won — even though items were not spatially separated by colour.
""")

In [ ]:
# ── Step 4: Gestalt in a real chart — proximity grouping with tips.csv ──────
# Grouped bar chart: tip amount by day and time (lunch vs dinner)
# Proximity principle: within-group gap < between-group gap signals which bars belong together

grouped = tips.groupby(['day', 'time'])['tip'].mean().unstack()
days = ['Thur', 'Fri', 'Sat', 'Sun']
grouped = grouped.reindex(days)

n_days = len(days)
x = np.arange(n_days)

# ── Version A: small within-group gap (proximity works) ────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

tight_w = 0.35
ax1.bar(x - tight_w/2, grouped['Lunch'],  width=tight_w, color='#4477aa', label='Lunch')
ax1.bar(x + tight_w/2, grouped['Dinner'], width=tight_w, color='#ee6677', label='Dinner')
ax1.set_xticks(x)
ax1.set_xticklabels(days)
ax1.set_ylabel('Mean tip ($)')
ax1.set_title('Proximity groups Lunch vs Dinner per day\n'
              'Gestalt principle used: PROXIMITY (within-gap < between-gap)', loc='left')
ax1.legend(frameon=False)
for side in ('top', 'right'):
    ax1.spines[side].set_visible(False)
ax1.set_ylim(0)

# ── Version B: equal gaps (proximity destroyed) ────────────────────────────
equal_w = 0.25
gap = 0.25
ax2.bar(x - equal_w - gap/4, grouped['Lunch'],  width=equal_w, color='#4477aa', label='Lunch')
ax2.bar(x + gap/4,           grouped['Dinner'], width=equal_w, color='#ee6677', label='Dinner')
ax2.set_xticks(x)
ax2.set_xticklabels(days)
ax2.set_title('Equal gaps — proximity destroyed; harder to compare Lunch vs Dinner\n'
              'Same data, same numbers — the grouping signal is gone', loc='left')
ax2.legend(frameon=False)
for side in ('top', 'right'):
    ax2.spines[side].set_visible(False)
ax2.set_ylim(0)

fig.suptitle('tips.csv: average tip by day and time — proximity as a design decision',
             x=0.02, ha='left')
plt.tight_layout()
plt.savefig('proximity_tips.png', dpi=300, bbox_inches='tight')
plt.show()

**What the reader loses in the equal-gap version:**

The data has not changed at all — every bar is identical. But without the proximity cue, the reader no longer knows *which grouping matters*. Are we comparing days? Are we comparing meal times? The chart no longer tells. In version A, the tight Lunch-Dinner pair says clearly: 'compare these two within each day'. In version B, the four bars drift apart and the reader must use the legend alone to reconstruct the pairing — additional working-memory load.

**Gestalt principle used:** *Proximity* — within-group gap (≈ 0 px of white space between bars) is smaller than between-group gap (≈ 20 px), so the brain naturally pairs the adjacent bars.

---

**Step 5: Where each Gestalt principle already lives in a chart:**

| Principle | Where it appears in charts |
|---|---|
| **Proximity** | A grouped bar chart: bars within a group are placed tight together; groups are separated by a wider gap. The spacing is a design decision about which comparison matters. |
| **Similarity** | A legend: every marker for 'Series A' uses the same blue colour, so they are read as belonging to the same category across all panels in a small multiple. |
| **Enclosure** | A small-multiples layout: each panel sits inside its own box or axis frame. The frame tells the brain that the data inside belongs to that facet's condition, not to the neighbour's. |
| **Closure** | A line chart: when one year's data is missing, the line has a gap. The brain often closes the gap and reads a continuous trend — a perception-driven interpolation the designer should flag with a note. |
| **Continuity** | A line chart's line itself: the eye follows the smooth path from left to right and attributes all points on that path to the same time series, even when multiple lines cross in the middle. |
| **Connection** | A table's rules (horizontal grid lines): the line visually joins the value in the left column with the values in every other column of the same row, signalling they belong to the same record. |

---

**Key question:** A legend forces the reader to hold a colour in working memory, carry it across the chart, and match it — this is a *similarity* operation mediated by working memory. Direct labelling instead uses *proximity* (the label is right next to the line) and *connection* (label and line share position). These are both cheaper: proximity and connection are resolved without attention, while a legend match requires a deliberate attentional shift. In terms of load: legend = extraneous load (the reader must do work to decode the encoding); direct label = germane load only (the label names the thing directly, with no translation step).